# SAOU center-spin demo

Train a shell-force CNEEP model from scratch, then drive the center spin at the same temperature. Maps use the homogeneous continuous-time midpoint medium-EP term; the pinning force and center site are excluded. Because trajectories use Euler steps, this is not the exact finite-step path log-ratio.

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import trange

starts = (Path.cwd().resolve(), (Path.cwd() / 'KNEEP').resolve())
ROOT = next(p for start in starts for p in (start, *start.parents)
            if (p / 'models/saou.py').is_file() and (p / 'shell_force.py').is_file())
(ROOT / 'results/.matplotlib').mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / 'results/.matplotlib'))

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

sys.path.insert(0, str(ROOT))

from models import saou
from models.saou import SAOUConfig
from utils.training import TrainingConfig, train_model

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

SOURCE_AMPLITUDE = 3.0
PIN_STRENGTH = 40.0
SOURCE_OMEGA = 0.5
BURN_STEPS = 10_000
N_TRANSITIONS = 10_000
SEED = 10_003
N_MOVIE_FRAMES = 240
INFERENCE_BATCH_SIZE = 16
FPS = 5


In [ ]:
TRAIN_L = 64
TRAIN_ENSEMBLES = 16
TRAIN_FRAMES = 500
N_ITER = 3000
TRAIN_SEED = 3

MAX_DISTANCE = 4
HIDDEN_CHANNELS = 64
HIDDEN_LAYERS = 3
VAL_RATIO = 0.2
TRAINING = TrainingConfig(alpha=-0.5, iterations=N_ITER, train_batch_size=16,
    validation_batch_size=16, learning_rate=1e-3, weight_decay=1e-4,
    gradient_clip=1.0, validate_every=N_ITER // 20)

A = 0.5
amplitudes = tuple(A / np.sqrt(k) for k in range(1, MAX_DISTANCE + 1))
omega0 = sum(amplitudes) + 0.2
SAOU_TRAIN = SAOUConfig(lattice_size=TRAIN_L, radii=tuple(range(1, MAX_DISTANCE + 1)),
    amplitudes=amplitudes, gamma=1.0, omega0=omega0, temperature=1.0, dt=1e-2)
ROLLOUT_T = SAOU_TRAIN.temperature
torch.manual_seed(TRAIN_SEED); np.random.seed(TRAIN_SEED)


## 1. Generate homogeneous training and validation trajectories


In [ ]:
video = saou.simulate_trajectories(
    SAOU_TRAIN, n_trajectories=TRAIN_ENSEMBLES, n_samples=TRAIN_FRAMES,
    burn_steps=BURN_STEPS, seed=TRAIN_SEED, simulation_device=DEVICE,
    storage_dtype=torch.float32, progress=True,
)
split = int(TRAIN_ENSEMBLES * (1.0 - VAL_RATIO))
train_video, val_video = video[:split], video[split:]

## 2. Train from scratch in this notebook (no pretrained checkpoint)


In [ ]:
result = train_model(train_video, val_video, TRAINING, model_seed=TRAIN_SEED, device=DEVICE,
    n_components=2, hidden_channels=HIDDEN_CHANNELS, hidden_layers=HIDDEN_LAYERS,
    max_distance=MAX_DISTANCE, activation='elu', progress=True)
model = result.model
config = SAOU_TRAIN
mean, std = (x.to(DEVICE, torch.float32) for x in (result.mean, result.std))
print(f'best iteration={result.best_iteration}, validation loss={result.best_validation_loss:.3e}')


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(result.history_iterations, result.train_losses, 'o-', ms=3, label='train')
plt.plot(result.history_iterations, result.validation_losses, 'o-', ms=3, label='validation')
plt.xlabel('iteration')
plt.ylabel('NEEP loss')
plt.title(f'{TRAIN_L}x{TRAIN_L} homogeneous SAOU training')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

## 3. Drive the center spin and evaluate the trained model


In [ ]:
L = config.lattice_size
cy = cx = L // 2
operators = saou._build_operators(config)
generator = torch.Generator(device=DEVICE).manual_seed(SEED)
noise_scale = np.sqrt(2.0 * ROLLOUT_T * config.dt)


def source_target(time):
    phase = SOURCE_OMEGA * time
    return torch.tensor(SOURCE_AMPLITUDE * np.array((np.cos(phase), np.sin(phase))), device=DEVICE)


@torch.no_grad()
def advance(field, time):
    force = torch.zeros_like(field)
    force[cy, cx] = PIN_STRENGTH * (source_target(time) - field[cy, cx])
    noise = torch.randn(field.shape, generator=generator, device=DEVICE, dtype=field.dtype)
    return field + (saou._drift(field, operators, config) + force) * config.dt + noise_scale * noise


@torch.no_grad()
def evaluate_pairs(pairs):
    predicted, ground_truth = [], []
    for start in range(0, len(pairs), INFERENCE_BATCH_SIZE):
        raw_pair = pairs[start:start + INFERENCE_BATCH_SIZE].to(DEVICE)
        predicted.append((model((raw_pair.float() - mean) / std, return_maps=True).sum(1) / TRAIN_L**2).cpu())

        first = raw_pair[:, 0].permute(0, 2, 3, 1)
        second = raw_pair[:, 1].permute(0, 2, 3, 1)
        force = saou.irreversible_velocity((first + second) / 2, operators, config.omega0) / ROLLOUT_T
        ground_truth.append((force * (second - first)).sum(-1).cpu())

    return torch.cat(predicted).numpy(), torch.cat(ground_truth).numpy()

print(f'device={DEVICE}, T_train={config.temperature:g}, T_rollout={ROLLOUT_T:g}')


In [ ]:
field = np.sqrt(ROLLOUT_T / config.gamma) * torch.randn((L, L, 2), generator=generator, device=DEVICE, dtype=torch.float64)

with torch.no_grad():
    for step in trange(BURN_STEPS, desc='center-spin burn-in'):
        field = advance(field, step * config.dt)

movie_ids = np.linspace(0, N_TRANSITIONS - 1, min(N_MOVIE_FRAMES, N_TRANSITIONS), dtype=int)
selected_ids = set(movie_ids)
pairs, spin_frames = [], []
previous = field.detach().clone()

with torch.no_grad():
    for transition_id in trange(N_TRANSITIONS, desc='center-spin rollout'):
        field = advance(field, (BURN_STEPS + transition_id) * config.dt)
        current = field.detach().clone()
        if transition_id in selected_ids:
            pairs.append(torch.stack((previous.permute(2, 0, 1), current.permute(2, 0, 1))))
            spin_frames.append(current.cpu())
        previous = current

pairs = torch.stack(pairs)
spin_frames = torch.stack(spin_frames).numpy()
pred_ep_frames, true_ep_frames = evaluate_pairs(pairs)
spin_angles = np.arctan2(spin_frames[..., 1], spin_frames[..., 0])
diff_ep_frames = pred_ep_frames - true_ep_frames
movie_times = (BURN_STEPS + movie_ids + 1) * config.dt

center_mask = np.zeros((L, L), bool); center_mask[cy, cx] = True
valid = ~center_mask
true_valid, pred_valid = (x[:, valid].ravel() for x in (true_ep_frames, pred_ep_frames))
pearson_r = float(np.corrcoef(true_valid, pred_valid)[0, 1])
relative_rmse = float(np.linalg.norm(pred_valid - true_valid) / max(np.linalg.norm(true_valid), 1e-12))
print(f'local EP: r={pearson_r:.4f}, relative RMSE={relative_rmse:.4f}')


In [ ]:
pooled_ep = np.concatenate((true_ep_frames[:, valid].ravel(), pred_ep_frames[:, valid].ravel()))
vmax_ep = max(1e-12, float(np.percentile(np.abs(pooled_ep), 99)))
vmax_diff = max(1e-12, float(np.percentile(np.abs(diff_ep_frames[:, valid]), 99)))
angle_cmap = plt.get_cmap('twilight').copy()
ep_cmap = plt.get_cmap('RdBu_r').copy()
ep_cmap.set_bad('lightgray')


def masked(frames, index):
    return np.ma.masked_where(center_mask, frames[index])


fig, axes = plt.subplots(1, 4, figsize=(16, 4.5), dpi=120)
spin_im = axes[0].imshow(
    spin_angles[0], origin='lower', cmap=angle_cmap, vmin=-np.pi, vmax=np.pi, interpolation='nearest'
)
true_im = axes[1].imshow(
    masked(true_ep_frames, 0), origin='lower', cmap=ep_cmap, vmin=-vmax_ep, vmax=vmax_ep, interpolation='nearest'
)
pred_im = axes[2].imshow(
    masked(pred_ep_frames, 0), origin='lower', cmap=ep_cmap, vmin=-vmax_ep, vmax=vmax_ep, interpolation='nearest'
)
diff_im = axes[3].imshow(
    masked(diff_ep_frames, 0), origin='lower', cmap=ep_cmap, vmin=-vmax_diff, vmax=vmax_diff, interpolation='nearest'
)

axes[0].set_title(r'Spin profile $\theta(x,y)$')
axes[1].set_title('Homogeneous-theory true local EP')
axes[2].set_title('Predicted local EP')
axes[3].set_title('Prediction - true')
axes[0].scatter([cx], [cy], s=60, facecolors='none', edgecolors='white', linewidths=1.2)
for axis in axes:
    axis.set_xticks([])
    axis.set_yticks([])
fig.colorbar(spin_im, ax=axes[0], fraction=0.046, label='spin angle')
fig.colorbar(true_im, ax=axes[1:3], fraction=0.026, pad=0.03, label='local EP increment')
fig.colorbar(diff_im, ax=axes[3], fraction=0.046, label='prediction - true')
title = fig.suptitle('')


def update(frame):
    spin_im.set_data(spin_angles[frame])
    true_im.set_data(masked(true_ep_frames, frame))
    pred_im.set_data(masked(pred_ep_frames, frame))
    diff_im.set_data(masked(diff_ep_frames, frame))
    title.set_text(
        f'Center spin: omega={SOURCE_OMEGA:g}, t={movie_times[frame]:.3f} | '
        f'EP r={pearson_r:.3f}, relative RMSE={relative_rmse:.3f}'
    )
    return spin_im, true_im, pred_im, diff_im, title


movie = FuncAnimation(
    fig, update, frames=len(movie_ids), interval=1000 / FPS, blit=False
)
fig.subplots_adjust(left=0.03, right=0.97, bottom=0.08, top=0.84, wspace=0.35)

plt.close(fig)
HTML(movie.to_jshtml())
